# Exp 01: Toy MLP

目标：
  1. 建立 eager vs compiled 的 timing 直觉
  2. 观察首次编译延迟 vs 稳态延迟
  3. 用 TORCH_LOGS=graph_code 看 Inductor 产出的 Triton kernel
  4. 用 counters 验证只编译了一次

特殊环境变量：

```bash
# 看 Inductor 产出的 Triton kernel（输出较多）
TORCH_LOGS=graph_code python exp01_toy_mlp.py

# 看 guard 树
TORCH_LOGS=guards python exp01_toy_mlp.py
```

硬件假设：NVIDIA GPU with CUDA, PyTorch 2.8

## 准备工作

### Import lib

必备库，注意 triton 和 torch 版本的配合

试了下默认 system_py310 config 有问题

In [2]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch._dynamo.utils import counters

### 模型定义

带有激活的三层 Linear，足够产生有意义的 kernel fusion，又简单到容易解读

In [3]:
class ToyMLP(nn.Module):
    """三层 MLP，足够产生有意义的 kernel fusion，又简单到容易解读"""

    def __init__(self, in_dim: int, hidden: int, out_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.gelu(self.fc1(x))
        x = F.gelu(self.fc2(x))
        return self.fc3(x)


### 计时工具

**首先为什么要写这个“工具”？**

请先思考，我们的目标是什么？其实是优化 GPU 执行的效率，因此需要 Profile 的时间是 GPU 的计算时间。那么如何在 GPU 上进行计时测速？

如果直接用 CPU 的计时器（如 `time.perf_counter()`）很容易不准，因为：

- 算子是以 异步 方式提交的：CPU 把 kernel 丢进队列就继续往下走，真正计算还在 GPU 上跑。
- 若不在合适位置 同步，你量的往往是“发命令有多快”，而不是“模型在 GPU 上算了多久”。

所以这里用 CUDA 自带的事件（Event） 在 GPU 时间轴 上打起点和终点，专门用来量 这次循环里、在 default stream 上排队的 GPU 工作 花了多少毫秒。


**其次，我们测量的核心是什么过程？**

通常训练中，GPU 上我们最关心的是 Forward 和 Backward 时间。

因此不调用 optimizer.step()，只测 前向 + 反向 的 kernel 时间，避免把优化器步长混进去。

这里 p.grad = None 是在每步后清梯度，这样下一次循环仍是“一步 fwd+bwd”的稳态，且不会因为累积梯度而扭曲含义。

**最后，如何实现准确测量**

首先我们利用 torch.cuda.Event，这是 CUDA 自带的 Event 记录系统，torch 只是调用。

具体用法可见



- start.record() / end.record()：在 同一条 CUDA stream 上插两个时间点；它们量的是 GPU 执行顺序里从“执行到 start”到“执行到 end”的间隔。

- enable_timing=True：打开事件计时能力。
start.elapsed_time(end)：返回这两个事件之间的 时间差，单位是毫秒（以 GPU 侧计为准，适合量 kernel 区间）。
和纯 CPU 计时的差别：Event 对齐的是 GPU 上实际跑完那批 work 的区间，比盲目用 perf_counter 更贴近 “fwd+bwd 在卡上” 的耗时（在单 stream、逻辑简单的场景下很常用）。


另外，我们利用 torch.cuda.synchronize() 来进行 CPU 和 GPU 的同步，实际上它会让当前设备上的 CUDA 工作做完再继续往下执行 CPU 逻辑。因此其实在默认的语义下，可以理解为：等到之前提交到该设备上的 work 都完成。

这里之所以用 两次 同步有典型原因：

1. start 之前同步一次：避免 上一轮/别的代码 还没跑完的 kernel 被算进本次计时区间，保证从“干净状态”再开始量。
2. end 之后同步一次：end.record() 只是把“记录结束事件”这条指令排队；若不同步，CPU 可能立刻去读 elapsed_time()，而 GPU 还没跑完中间那段循环。同步后，才能保证 start 到 end 之间的 GPU 活都真的完成了，读出来的时间才有意义。

In [4]:
def time_step(fn, x, y, n: int) -> float:
    """Measure the average wall time (ms) per step for n steps of forward+backward+zero_grad."""
    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize()
    start.record()
    for _ in range(n):
        loss = F.mse_loss(fn(x), y)
        loss.backward()
        # 不调 optimizer.step，专注测 fwd+bwd kernel 时间
        for p in fn.parameters():
            p.grad = None
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / n  # ms/step


### 其他定义

In [5]:
DEVICE    = "cuda"
DTYPE     = torch.bfloat16   # 需要选一个适配的 GPU，比如 L20 原生支持，因此我们大概率无需 GradScaler
BATCH     = 256
IN_DIM    = 1024
HIDDEN    = 4096
OUT_DIM   = 512
WARMUP    = 5                 # 稳定 GPU clock 的预热步数
MEASURE   = 50                # 正式计时步数

In [6]:
torch.manual_seed(42)

model = ToyMLP(IN_DIM, HIDDEN, OUT_DIM).to(DEVICE, dtype=DTYPE)
x = torch.randn(BATCH, IN_DIM,  device=DEVICE, dtype=DTYPE)
y = torch.randn(BATCH, OUT_DIM, device=DEVICE, dtype=DTYPE)

model

ToyMLP(
  (fc1): Linear(in_features=1024, out_features=4096, bias=True)
  (fc2): Linear(in_features=4096, out_features=4096, bias=True)
  (fc3): Linear(in_features=4096, out_features=512, bias=True)
)

## 实验

### Eager Mode - Baseline

经典 Pytorch Eager Mode，作为 baseline（无 compile）

In [18]:
warmup_ms = time_step(model, x, y, WARMUP)
print(f"Eager Mode: warmup {warmup_ms:.3f} ms/step")

eager_ms = time_step(model, x, y, MEASURE)
print(f"Eager Mode: fwd+bwd {eager_ms:.3f} ms/step")

Eager Mode: warmup 1.194 ms/step
Eager Mode: fwd+bwd 0.929 ms/step


### First Compile Overhead

使用 torch.compile 的时候，第一次的执行是在编译的过程，自然会有 overhead，需要单独测试

In [8]:
# 重建一个相同结构的模型，避免 eager warm-up 状态干扰
model_c = ToyMLP(IN_DIM, HIDDEN, OUT_DIM).to(DEVICE, dtype=DTYPE)
model_c.load_state_dict(model.state_dict())

counters.clear()

compiled = torch.compile(model_c, mode="default", fullgraph=False)

t0 = time.perf_counter()
loss = F.mse_loss(compiled(x), y)
loss.backward()
for p in model_c.parameters(): p.grad = None
torch.cuda.synchronize()
first_call_ms = (time.perf_counter() - t0) * 1000

print(f"  首次 call（含编译）: {first_call_ms:.1f} ms")
print(f"  unique_graphs after 1st call: {counters['stats']['unique_graphs']}")

  首次 call（含编译）: 1735.3 ms
  unique_graphs after 1st call: 1


### After Compile

测试编译后稳态的结果

In [22]:
warmup_compiled_ms = time_step(compiled, x, y, WARMUP)
print(f"Compiled Mode: warmup {warmup_compiled_ms:.3f} ms/step")

compiled_ms = time_step(compiled, x, y, MEASURE)
print(f"Compiled Mode: fwd+bwd: {compiled_ms:.3f} ms/step")

print('-'*50)
speedup = eager_ms / compiled_ms
print(f"Compield vs Eager: {speedup:.2f}x")

# unique_graphs 应该还是 1（shape 没变，没有 recompile）
print(f"unique_graphs after {WARMUP+MEASURE} calls: {counters['stats']['unique_graphs']}")


Compiled Mode: warmup 1.433 ms/step
Compiled Mode: fwd+bwd: 1.197 ms/step
--------------------------------------------------
Compield vs Eager: 0.78x
unique_graphs after 55 calls: 1


### Different Compile Mode

目前 `torch.compile(..., mode=...)` 常见/官方主要就是这几个：

```python
"default"  # `mode=None` 基本等价于默认模式
"reduce-overhead"
"max-autotune"
"max-autotune-no-cudagraphs"
```

不同 Mode 特征总览

| mode | 核心目标 | 编译时间 | 运行速度潜力 | CUDA Graphs | 适合场景 | 主要特点 |
|---|---|---:|---:|---|---|---|
| `default` | 平衡编译开销和运行性能 | 中 | 中 | 不特别激进 | 首次尝试、教学 baseline、一般模型 | 默认优化策略，最稳妥，适合先验证能否跑通 |
| `reduce-overhead` | 减少 Python / kernel launch overhead | 中到高 | 中到高 | 通常更积极使用 | 小模型、小 batch、inference、shape 稳定场景 | 重点减少 CPU 提交和调度开销，不一定显著优化单个 kernel |
| `max-autotune` | 追求更高 steady-state 性能 | 高 | 高 | 通常使用 | 长时间训练、benchmark、production inference、shape 稳定场景 | 编译时尝试更多 kernel / GEMM 方案，用更长编译时间换更快运行 |
| `max-autotune-no-cudagraphs` | autotune 但禁用 CUDA Graphs | 高 | 高 | 不使用 | 动态 shape、CUDA Graphs 不稳定、显存压力较大场景 | 保留 autotune 优化，但避免 CUDA Graphs 带来的限制和额外约束 |

In [25]:
modes = [
    "default",
    "reduce-overhead",
    "max-autotune",
    "max-autotune-no-cudagraphs",   # 跳过 cudagraphs 避免静态 shape 限制
]

results = {}
for mode in modes:
    m = ToyMLP(IN_DIM, HIDDEN, OUT_DIM).to(DEVICE, dtype=DTYPE)
    m.load_state_dict(model.state_dict())
    c = torch.compile(m, mode=mode, fullgraph=False)

    t0 = time.perf_counter()
    F.mse_loss(c(x), y).backward()
    for p in m.parameters(): p.grad = None
    torch.cuda.synchronize()
    compile_ms = (time.perf_counter() - t0) * 1000

    warmup_ms = time_step(c, x, y, WARMUP)

    steady_ms = time_step(c, x, y, MEASURE)
    results[mode] = (compile_ms, steady_ms)
    print(f"  [{mode:36s}]  首次={compile_ms:7.0f}ms  稳态={steady_ms:.3f}ms")

  [default                             ]  首次=      2ms  稳态=1.150ms
  [reduce-overhead                     ]  首次=     12ms  稳态=0.840ms
  [max-autotune                        ]  首次=     11ms  稳态=0.842ms
  [max-autotune-no-cudagraphs          ]  首次=      3ms  稳态=1.161ms


In [26]:
# ── 5. 小结 ──────────────────────────────────────────────────────────────
print(f"  eager baseline : {eager_ms:.3f} ms/step")
for mode, (ctime, stime) in results.items():
    sx = eager_ms / stime
    print(f"  {mode:36s}: 稳态 {stime:.3f} ms  ({sx:.2f}x)  首次编译 {ctime:.0f} ms")


  eager baseline : 0.929 ms/step
  default                             : 稳态 1.150 ms  (0.81x)  首次编译 2 ms
  reduce-overhead                     : 稳态 0.840 ms  (1.11x)  首次编译 12 ms
  max-autotune                        : 稳态 0.842 ms  (1.10x)  首次编译 11 ms
  max-autotune-no-cudagraphs          : 稳态 1.161 ms  (0.80x)  首次编译 3 ms


关键结论：
- 首次 call 包含 第一次 compiled call 包含 trace、graph capture、AOTAutograd、Inductor codegen、kernel 编译/缓存等，应该单独看。不能把首次耗时和 eager 单步耗时直接比。
- 稳态下 unique_graphs == 1，说明 shape 不变时不会 recompile
- max-autotune 首次编译明显更长，但稳态 GEMM 选了最优 Triton tile 配置
- reduce-overhead 靠 CUDA Graphs 降低 CPU dispatch 开销，对小 batch 效果最显著
- compile 不一定比 eager 快。比如 default mode 反而慢到 0.81x。这很正常，compile 引入的图执行、调度、autograd partition、生成 kernel 选择等不一定总能胜过 eager/cuBLAS/cuDNN 的成熟路径。
- 这个模型的主要计算是大 Linear/GEMM，eager 已经很强。三层 MLP 的主体是 nn.Linear，底层通常走 cuBLASLt 之类高度优化库。torch.compile 对这种“已经由大 GEMM 主导”的模型，额外可融合的 elementwise 部分有限，所以收益可能不大。